# VitaCall ML Engineering & Ops
---

In [ ]:
# !pip install pyspark scikit-learn pandas numpy mlflow fastapi uvicorn requests

## 1. Datapipeline (Week 6)

### 1.1 Data-ingestie
Ruwe IMDb-reviews downloaden en opslaan als Parquet.

In [ ]:
import os, sys
import pandas as pd
sys.path.insert(0, '..')
from pipeline.data import download_imdb, ingest_imdb

base_dir   = '../data'
bronze_dir = os.path.join(base_dir, 'bronze', 'imdb')
os.makedirs(bronze_dir, exist_ok=True)

imdb_dir = download_imdb(base_dir)
ingest_imdb(imdb_dir, bronze_dir)

df_bronze = pd.read_parquet(os.path.join(bronze_dir, 'imdb.parquet'))
print(f'Rijen: {len(df_bronze)}')
df_bronze.head(3)

### 1.2 Data cleaning
HTML-tags verwijderen, lege teksten en duplicaten eruit filteren.

In [ ]:
from pipeline.data import clean_reviews

silver_dir = os.path.join(base_dir, 'silver', 'imdb')
clean_reviews(os.path.join(bronze_dir, 'imdb.parquet'), silver_dir)

df_silver = pd.read_parquet(silver_dir)
print(f'Silver rijen: {len(df_silver)}')
df_silver[['text_clean', 'label', 'split']].head(3)

### 1.3 Feature engineering
Stratified train/val/test splits met PySpark.

In [ ]:
from pipeline.data import get_spark, create_features

gold_dir = os.path.join(base_dir, 'gold', 'imdb')
os.makedirs(gold_dir, exist_ok=True)

spark = get_spark()
spark.sparkContext.setLogLevel('WARN')
create_features(spark, silver_dir, gold_dir)
spark.stop()

df_gold = pd.read_parquet(gold_dir)
print(df_gold.groupby(['split', 'label']).size().reset_index(name='count').to_string())
df_gold.head(3)

In [ ]:
from pipeline.data import train

model_path = '../pipeline/sentiment_model.pkl'
df_tr  = df_gold[df_gold['split'] == 'train']
df_val = df_gold[df_gold['split'] == 'val']
df_te  = df_gold[df_gold['split'] == 'test']

metrics = train(
    df_tr['text_clean'].tolist(),  df_tr['label'].tolist(), model_path,
    val_texts=df_val['text_clean'].tolist(), val_labels=df_val['label'].tolist()
)
print('Validatie metrics:', metrics)

---

## Bronvermelding

- IMDb dataset: Maas, A.L. et al. (2011). *Learning Word Vectors for Sentiment Analysis.*
- scikit-learn: Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python.*
- MLflow: https://mlflow.org/
- PySpark: https://spark.apache.org/docs/latest/api/python/
- FastAPI: https://fastapi.tiangolo.com/
- Vosk: https://alphacephei.com/vosk/